In [1]:
from scipy.io import loadmat
import cv2
import numpy as np
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor, as_completed
from mtcnn import MTCNN
import fcntl
import shutil
import math
from collections import Counter
import mplcursors
import albumentations as A
import seaborn as sns
import random
import csv
import sys
import matplotlib.pyplot as plt
from collections import Counter
import pandas as pd
import os
from tqdm import tqdm

I0000 00:00:1792857131.486171   34675 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792857131.996776   34675 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792857134.670494   34675 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
db_folder = '../Photometric_DB_Minh'
db_process = '../Photometric_DB'
total_cores = os.cpu_count() - 2

In [3]:
session_paths = []

for id in os.listdir(db_folder):
    if id.isdigit():
        id_path = os.path.join(db_folder, id)
        for session in os.listdir(id_path):
            session_path = os.path.join(id_path, session)
            if os.path.isdir(session_path):
                session_paths.append(session_path)

In [13]:
# Chứa các file nếu xử lý bị lỗi
error_file = '2.error_generate_3d_images.txt'

def write_errors(error_messages):
    if error_messages:
        with open(error_file, 'a', encoding='utf-8') as f:
            f.write('\n'.join(error_messages) + '\n')
            
def create_3d_images(session_path):
    id_dir = os.path.basename(os.path.dirname(session_path))
    sn_path = os.path.join(session_path, 'sn.mat')
    z_path = os.path.join(session_path, 'z.mat')
    albedo_path = os.path.join(session_path, 'albedo.mat')
    
    error_messages = []

    if os.path.exists(sn_path) and os.path.exists(z_path) and os.path.exists(albedo_path):
        try:
            # Load dữ liệu từ các file .mat
            sn_data = loadmat(sn_path)
            z_data = loadmat(z_path)
            albedo_data = loadmat(albedo_path)
            
            # Trích xuất các thông tin cần thiết từ mat file
            px = sn_data['px']
            py = sn_data['py']
            pz = sn_data['pz']
            z = z_data['z']
            albedo = albedo_data['a']
            
            # Save các ảnh normal map
            try:
                normal_map = cv2.merge([pz, -px, py])
                normal_map = (normal_map + 1)/2
                normal_map_16bit = np.round(normal_map * 65535).astype(np.uint16)
                normal_map_path = os.path.join(session_path, 'normalmap.png')
                cv2.imwrite(normal_map_path, normal_map_16bit)
            except Exception as e:
                error_messages.append(f"{id_dir}: Error saving normal map: {repr(e)}")

            # Save các ảnh depth map
            try:
                depth_map_path = os.path.join(session_path, 'depthmap.png')
                cv2.imwrite(depth_map_path, z)
            except Exception as e:
                error_messages.append(f"{id_dir}: Error saving depth map: {repr(e)}")

            # Save các ảnh albedo map
            try:
                albedo_map_path = os.path.join(session_path, 'albedo.png')
                cv2.imwrite(albedo_map_path, albedo)
            except Exception as e:
                error_messages.append(f"{id_dir}: Error saving albedo map: {repr(e)}")

            if error_messages:
                error_str = "\n".join(error_messages)
                write_errors(error_messages)
                return error_str
            return None

        except Exception as e:
            error_str = f"{id_dir}: Error loading .mat files: {repr(e)}"
            write_errors([error_str])
            return error_str
    else:
        error_str = f"{id_dir}: Missing files"
        write_errors([error_str])
        return error_str

In [14]:
with ProcessPoolExecutor(max_workers=total_cores) as executor:
    list(tqdm(executor.map(create_3d_images, session_paths), total=len(session_paths)))

  0%|          | 0/3159 [00:00<?, ?it/s][ WARN:0@1739.330] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
[ WARN:0@1739.366] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
[ WARN:0@1739.369] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
[ WARN:0@1739.395] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
[ WARN:0@1739.430] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
[ WARN:0@1739.446] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
[ WARN:0@1739.482] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
[ WARN:0@1739.484] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallb

# Copy File sang folder mới

In [4]:
import os
import shutil
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

db_folder = '../Photometric_DB_Minh'
db_process = '../Photometric_DB'

total_cores = max(1, (os.cpu_count() or 1) - 2)

file_name = [
    'im0.bmp', 'im1.bmp', 'im2.bmp', 'im3.bmp',
    'LightSource.m', 'metadata.txt', 'sn.mat', 'z.mat', 'albedo.mat',
    'normalmap.png', 'depthmap.png', 'albedo.png'
]

os.makedirs(db_process, exist_ok=True)

session_paths = []

for id_dir in os.listdir(db_folder):
    id_src = os.path.join(db_folder, id_dir)

    if not os.path.isdir(id_src):
        continue

    for session_dir in os.listdir(id_src):
        session_path = os.path.join(id_src, session_dir)

        if os.path.isdir(session_path):
            session_paths.append(session_path)
            
def process_session(session_path):
    # Lấy ID và tên session từ đường dẫn
    id_dir = os.path.basename(os.path.dirname(session_path))
    session_dir = os.path.basename(session_path)

    # Tạo đường dẫn đích theo cấu trúc ID/session
    session_dst = os.path.join(db_process, id_dir, session_dir)
    os.makedirs(session_dst, exist_ok=True)

    copied = 0
    missing = []

    # Copy các file được chỉ định
    for filename in file_name:
        src = os.path.join(session_path, filename)
        dst = os.path.join(session_dst, filename)

        if os.path.isfile(src):
            try:
                shutil.copy2(src, dst)
                copied += 1
            except Exception as e:
                missing.append(f'Copy error: {src}: {repr(e)}')
        else:
            missing.append(f'Missing: {src}')

    return copied, missing

In [5]:
with ProcessPoolExecutor(max_workers=total_cores) as executor:
    list(tqdm(executor.map(process_session, session_paths), total=len(session_paths)))

100%|██████████| 3185/3185 [08:32<00:00,  6.22it/s]
